# LangChain con OpenAI

In [ ]:
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

In [6]:
http_client = httpx.Client(verify=False)

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0.7,
    http_client=http_client
)

In [8]:
pregunta = "¿Cuál es la capital de Francia?. Solo dame la respuesta sin explicaciones."

In [9]:
respuesta = llm.invoke(pregunta)

In [11]:
respuesta

AIMessage(content='París.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 23, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b4abb0dc92', 'id': 'chatcmpl-Dww7LEwIeniBQNa6jn5ZfMmoHEUOy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f1f6f-05b6-7593-a44a-1a5b9837c0c1-0', usage_metadata={'input_tokens': 23, 'output_tokens': 3, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# LangChain con Google

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
llm_google = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.7
)

In [17]:
pregunta_google = "¿Cuál es la capital de Francia?. Solo dame la respuesta sin explicaciones."

In [18]:
pregunta_google = llm_google.invoke(pregunta)

In [19]:
pregunta_google

AIMessage(content='París', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []}, id='run--019f1f75-8e73-78e2-bdbb-4a58feaa03f7-0', usage_metadata={'input_tokens': 17, 'output_tokens': 34, 'total_tokens': 51, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 32}})

# LangChain avanzado usando LangChain Expression Language (LCEL)

In [21]:
from dotenv import load_dotenv,find_dotenv
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

http_client = httpx.Client(verify=False)

In [22]:
load_dotenv(find_dotenv())

True

In [23]:
chat = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0.7,
    http_client=http_client
)

In [24]:
plantilla = PromptTemplate(
    input_variables=["pais","nombre"],
    template="Saluda al usuario con su nombre. \nNombre del usuario {nombre}\n¿Cuál es la capital de {pais}? Solo dame la respuesta sin explicaciones."
)

#### Aqui está la magia de LCEL, podemos encadenar la plantilla con el modelo de lenguaje de una forma muy sencilla y elegante, simplemente usando el operador "|" para encadenar la plantilla con el modelo de lenguaje.

In [25]:
cadena = plantilla | chat

In [26]:
resultado =cadena.invoke({"nombre":"Jonatan", "pais":"Francia"})

In [29]:
print(resultado.content)

¡Hola, Jonatan! La capital de Francia es París.


# Runnables

Los Runnables son objetos que representan una función o una operación que se puede ejecutar.

En este ejemplo, creamos dos Runnables: uno para generar un texto a partir de un número y otro para duplicar ese texto.


In [1]:
from langchain_core.runnables import RunnableLambda

In [2]:
# El primer Runnable, `paso1`, utiliza una función lambda para convertir un número en una cadena de texto con el formato "Número {x}".
paso1 = RunnableLambda(lambda x: f"Número {x}")

In [3]:
# El segundo Runnable, `paso2`, utiliza una función definida previamente, `duplicar_texto`, que toma un texto y lo duplica en una lista.
def duplicar_texto(texto):
    return [texto] * 2

paso2 = RunnableLambda(duplicar_texto)

In [ ]:
# Luego, combinamos ambos Runnables utilizando el operador `|`, lo que crea una nueva cadena de Runnables que ejecutará ambos pasos en secuencia.
# Cuando invocamos esta cadena con el número 43, primero se generará el texto "Número 43" y luego se duplicará en una lista.
cadena = paso1 | paso2

In [5]:
# Finalmente, invocamos la cadena de Runnables con el número 43 y almacenamos el resultado en la variable `resultado`.
resultado = cadena.invoke(43)

In [6]:
# Imprimimos el resultado, que será una lista con dos elementos, ambos siendo "Número 43".
print(resultado)

['Número 43', 'Número 43']


# Prompt template

Código que muestra como usar PromptTemplate y como probarlo antes de enviarlo al LLM con la funcion format. 

In [8]:
from langchain_core.prompts import PromptTemplate

In [9]:
template = "Eres un experto en marketing. Sugiere un eslogan para un producto que es: {producto}."

In [10]:
prompt = PromptTemplate(
    template=template,
    input_variables=["producto"]
)

In [11]:
prompt_completo = prompt.format(producto="una bebida energética saludable")

In [12]:
print(prompt_completo)

Eres un experto en marketing. Sugiere un eslogan para un producto que es: una bebida energética saludable.


# Prompt template avanzado

In [16]:
import httpx
from dotenv import load_dotenv,find_dotenv

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [18]:
load_dotenv(find_dotenv())
http_client = httpx.Client(verify=False)

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0.7,
    http_client=http_client
)

In [20]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un traductor del español al inglés muy preciso y profesional."),
    ("human", "Traduce el siguiente texto: {texto}")
])

In [21]:
mensajes = chat_prompt.format_messages(texto="Hola, ¿cómo estás?")

In [22]:
for m in mensajes:
    print(f"{type(m)}: {m.content}")

<class 'langchain_core.messages.system.SystemMessage'>: Eres un traductor del español al inglés muy preciso y profesional.
<class 'langchain_core.messages.human.HumanMessage'>: Traduce el siguiente texto: Hola, ¿cómo estás?


In [23]:
cadena = chat_prompt | llm
resultado = cadena.invoke({"texto":"Hola, ¿cómo estás?"})
print(f"Respuesta del modelo: {resultado.content}")

Respuesta del modelo: Hello, how are you?


# Proyecto:
scr/00.proyecto_modulo_2

------------

# Messages placeholders

Los MessagesPlaceholder permiten incluir una lista de mensajes en el prompt, manteniendo el contexto de la conversación.

In [24]:
# En este ejemplo, se utiliza para incluir el historial de la conversación antes de la pregunta actual del usuario.
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [25]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil que mantiene el contexto de la conversación."),
    MessagesPlaceholder(variable_name="historial"),
    ("human", "{pregunta_actual}"),
])

In [26]:
historial_conversacion = [
    HumanMessage(content="¿Cuál es la capital de Francia?"),
    AIMessage(content="La capital de Francia es París."),
    HumanMessage(content="¿Y cuántos habitantes tiene?"),
    AIMessage(content="París tiene aproximadamente 2.2 millones de habitantes en la ciudad propiamente dicha.")
]

In [27]:
mensajes = chat_prompt.format_messages(
    historial = historial_conversacion,
    pregunta_actual = "¿Puedes decirme algo interesante de su arquitectura?"
)

In [28]:
for mensaje in mensajes:
    print(f"{mensaje.type}: {mensaje.content}")

system: Eres un asistente útil que mantiene el contexto de la conversación.
human: ¿Cuál es la capital de Francia?
ai: La capital de Francia es París.
human: ¿Y cuántos habitantes tiene?
ai: París tiene aproximadamente 2.2 millones de habitantes en la ciudad propiamente dicha.
human: ¿Puedes decirme algo interesante de su arquitectura?


## Lectura: MessagesPlaceholder y Few-Shot Examples

## ¿Qué es el In-Context Learning (ICL)?

El In-Context Learning o Aprendizaje en Contexto es la capacidad de los modelos de lenguaje de aprender nuevas tareas o patrones simplemente proporcionándoles ejemplos dentro del mismo prompt, sin necesidad de entrenar o ajustar el modelo.

Es como mostrarle a alguien cómo hacer algo dándole ejemplos directos:

> "Mira, así se hace esto... y esto otro... ¿ahora puedes hacer tú algo similar?"

---

## ¿Qué son los Few-Shot Examples?

Los Few-Shot Examples (ejemplos de pocos intentos) son una técnica de ICL donde proporcionamos al modelo entre 2-10 ejemplos de la tarea que queremos que realice. Estos ejemplos sirven como "entrenamiento instantáneo".

**Estructura típica:**

* **Sistema:** Instrucciones generales
* **Ejemplo 1:** Input → Output esperado
* **Ejemplo 2:** Input → Output esperado  
* **Ejemplo 3:** Input → Output esperado
* **Pregunta real:** Input actual → ?

---

## MessagesPlaceholder: La Herramienta Perfecta

`MessagesPlaceholder` es ideal para few-shot examples porque:

* ✅ **Mantiene la estructura de mensajes** (Human/AI)
* ✅ **Permite insertar múltiples ejemplos** dinámicamente
* ✅ **El modelo entiende mejor** el formato de conversación
* ✅ **Fácil de reutilizar** para diferentes tareas

---

## Evolución del Código: De Historial a Few-Shot

### Código Base (Historial de Conversación)

```python
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
 
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil que mantiene el contexto de la conversación."),
    MessagesPlaceholder(variable_name="historial"),
    ("human", "Usuario: {pregunta_actual}")
])
 
# Simulamos un historial de conversación
historial_conversacion = [
    HumanMessage(content="Usuario: ¿Cuál es la capital de Francia?"),
    AIMessage(content="IA: La capital de Francia es París."),
    HumanMessage(content="Usuario: ¿Y cuántos habitantes tiene?"),
    AIMessage(content="IA: París tiene aproximadamente 2.2 millones de habitantes en la ciudad propiamente dicha.")
]
```

### Código Evolucionado (Few-Shot Examples)

```python
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
 
# Template para clasificación de sentimientos con few-shot examples
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en análisis de sentimientos. Clasifica cada texto como: POSITIVO, NEGATIVO o NEUTRO."),
    MessagesPlaceholder(variable_name="ejemplos"),
    ("human", "Texto a analizar: {texto_usuario}")
])
 
# Few-shot examples para análisis de sentimientos
ejemplos_sentimientos = [
    HumanMessage(content="Texto a analizar: Me encanta este producto, es increíble"),
    AIMessage(content="POSITIVO"),
    HumanMessage(content="Texto a analizar: El servicio fue terrible, muy decepcionante"),
    AIMessage(content="NEGATIVO"),
    HumanMessage(content="Texto a analizar: El clima está nublado hoy"),
    AIMessage(content="NEUTRO")
]
 
# Generar el prompt con los ejemplos
mensajes = chat_prompt.format_messages(
    ejemplos=ejemplos_sentimientos,
    texto_usuario="¡Qué día tan maravilloso!"
)
 
# Ver el resultado
for i, m in enumerate(mensajes):
    print(f"Mensaje {i+1} ({m.__class__.__name__}):")
    print(m.content)
    print("-" * 40)

```
### Salida del Código:

```text
Mensaje 1 (SystemMessage):
Eres un experto en análisis de sentimientos. Clasifica cada texto como: POSITIVO, NEGATIVO o NEUTRO.
----------------------------------------
Mensaje 2 (HumanMessage):
Texto a analizar: Me encanta este producto, es increíble
----------------------------------------
Mensaje 3 (AIMessage):
POSITIVO
----------------------------------------
Mensaje 4 (HumanMessage):
Texto a analizar: El servicio fue terrible, muy decepcionante
----------------------------------------
Mensaje 5 (AIMessage):
NEGATIVO
----------------------------------------
Mensaje 6 (HumanMessage):
Texto a analizar: El clima está nublado hoy
----------------------------------------
Mensaje 7 (AIMessage):
NEUTRO
----------------------------------------
Mensaje 8 (HumanMessage):
Texto a analizar: ¡Qué día tan maravilloso!
----------------------------------------
```

Ventajas de Este Enfoque

Con MessagesPlaceholder:

✅ Estructura clara: Cada ejemplo mantiene su rol (Human/AI)
✅ Escalable: Fácil añadir/quitar ejemplos
✅ Reutilizable: Cambiar ejemplos = Nueva tarea
✅ Natural: El modelo entiende el formato conversacional

Sin MessagesPlaceholder (texto plano):

❌ Menos claro: Todo mezclado en un string
❌ Más errores: El modelo puede confundirse
❌ Difícil mantenimiento: Cambios requieren reescribir todo
❌ Menos efectivo: Pierde la estructura conversacional



Otros Ejemplos de Uso

1. Extracción de Información

```json
ejemplos_extraccion = [
    HumanMessage(content="Texto: Juan Pérez trabaja en Google como ingeniero desde 2020"),
    AIMessage(content="Nombre: Juan Pérez, Empresa: Google, Puesto: ingeniero, Año: 2020"),
    HumanMessage(content="Texto: María Silva es doctora en el Hospital Central"),
    AIMessage(content="Nombre: María Silva, Empresa: Hospital Central, Puesto: doctora, Año: N/A")
]
```

2. Traducción con Estilo

```json
ejemplos_traduccion = [
    HumanMessage(content="Formal en inglés: Good morning, how are you today?"),
    AIMessage(content="Casual en español: ¡Hola! ¿Qué tal?"),
    HumanMessage(content="Formal en inglés: I would like to schedule a meeting"),
    AIMessage(content="Casual en español: ¿Podemos quedar?")
]
```

Mejores Prácticas

2-5 ejemplos: Suficiente para mostrar el patrón, no demasiados

Ejemplos diversos: Cubrir diferentes casos/variaciones

Formato consistente: Mismo patrón en todos los ejemplos

Ejemplos de calidad: Outputs perfectos como referencia



Conclusión

MessagesPlaceholder transforma los few-shot examples de una técnica básica a una herramienta poderosa y flexible. La estructura clara de mensajes Human/AI hace que el modelo comprenda mejor qué esperamos, resultando en respuestas más precisas y consistentes.

Recuerda: No es solo insertar ejemplos, es enseñarle al modelo a través de la conversación. ¡Esa es la magia del In-Context Learning! ✨

# Rol Prompt Template

In [29]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [30]:
plantilla_sistema = SystemMessagePromptTemplate.from_template("" \
    "Eres un {rol} especializado en {especialidad}. Responde de manera {tono}"
)

In [31]:
plantilla_humano = HumanMessagePromptTemplate.from_template(
    "Mi pregunta sobre {tema} es: {pregunta}"
)

In [32]:
chat_prompt = ChatPromptTemplate.from_messages([
    plantilla_sistema,
    plantilla_humano
])

In [33]:
mensajes = chat_prompt.format_messages(
    rol = "nutricionista",
    especialidad = "dietas veganas",
    tono = "profesional, pero accesible",
    tema = "proteinas vegetales",
    pregunta = "¿Cuáles son las mejores fuentes de proteina vegana para un atleta profesional?"
)

In [34]:
for m in mensajes:
    print(m.content)

Eres un nutricionista especializado en dietas veganas. Responde de manera profesional, pero accesible
Mi pregunta sobre proteinas vegetales es: ¿Cuáles son las mejores fuentes de proteina vegana para un atleta profesional?


# Output parsers

## PydanticOutputParser: El Método Clásico para Salidas Estructuradas

Antes de que LangChain introdujera `with_structured_output()`, los desarrolladores usaban **`PydanticOutputParser`** para obtener respuestas estructuradas de los modelos de lenguaje. Aunque es un método más verboso, sigue siendo ampliamente utilizado y te ayudará a entender los fundamentos del *structured output*.

---

## ¿Qué es PydanticOutputParser?

`PydanticOutputParser` es una clase que convierte las respuestas de texto libre de un LLM en objetos Python estructurados usando modelos de Pydantic. Funciona añadiendo instrucciones específicas al prompt y luego parseando la respuesta del modelo.

---

## Instalación de Dependencias

```bash
pip install langchain langchain-openai pydantic
```

---

## Ejemplo Completo Paso a Paso

### Paso 1: Definir el Modelo Pydantic

```python
from pydantic import BaseModel, Field

class AnalisisTexto(BaseModel):
    resumen: str = Field(description="Resumen breve del texto")
    sentimiento: str = Field(description="Sentimiento del texto (Positivo, neutro o negativo)")
    palabras_clave: list[str] = Field(description="Lista de palabras clave del texto")
```

**Explicación:**

- **`BaseModel`**: Clase base de Pydantic para crear modelos de datos.
- **`Field()`**: Permite añadir descripciones y validaciones.
- Las descripciones son **cruciales**: el parser las usa para generar instrucciones.

---

### Paso 2: Crear el Parser

```python
from langchain.output_parsers import PydanticOutputParser

# Crear el parser con nuestro modelo
parser = PydanticOutputParser(pydantic_object=AnalisisTexto)

# Ver las instrucciones que genera automáticamente
print("Instrucciones generadas:")
print(parser.get_format_instructions())
```

---

### Paso 3: Crear el Prompt Template

```python
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""Eres un experto analista de texto. Analiza el siguiente texto con mucho cuidado y proporciona un análisis detallado.

{format_instructions}

Texto a analizar:
{texto}

Análisis:""",
    input_variables=["texto"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
```

**Puntos importantes:**

- `{format_instructions}` se sustituye automáticamente con las instrucciones del parser.
- `partial_variables` permite pre-llenar variables del template.
- El prompt debe ser claro sobre qué esperas del modelo.

---

### Paso 4: Configurar el LLM

```python
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,  # Baja temperatura para respuestas más consistentes
)
```

---

### Paso 5: Crear la Cadena de Procesamiento

```python
# Crear la cadena: prompt → LLM → parser
chain = prompt | llm | parser
```

**¿Qué hace cada paso?**

- **Prompt**: Genera el mensaje completo con instrucciones.
- **LLM**: Procesa el prompt y genera una respuesta.
- **Parser**: Convierte la respuesta en un objeto Pydantic.

---

### Paso 6: Ejecutar el Análisis

```python
# Texto de ejemplo
texto_prueba = """
La nueva película de ciencia ficción 'Estrella Galáctica' es absolutamente 
espectacular. Los efectos visuales son impresionantes y la trama mantiene 
la tensión durante toda la película. Los actores principales entregan 
actuaciones convincentes que realmente te hacen creer en este mundo futurista.
Sin duda una de las mejores películas del año.
"""

try:
    # Invocar la cadena
    resultado = chain.invoke({"texto": texto_prueba})

    # Acceder a los datos
    print("=== RESULTADO DEL ANÁLISIS ===")
    print(f"Resumen: {resultado.resumen}")
    print(f"Sentimiento: {resultado.sentimiento}")
    print(f"Palabras clave: {', '.join(resultado.palabras_clave)}")

    # Exportar como JSON
    print("\n=== JSON RESULTANTE ===")
    print(resultado.model_dump_json(indent=2))

    # Exportar como diccionario
    dict_resultado = resultado.model_dump()
    print(f"\nTipo de objeto: {type(resultado)}")
    print(f"Tipo de diccionario: {type(dict_resultado)}")

except Exception as e:
    print(f"❌ Error durante el procesamiento: {e}")
```

---

## Código Completo Funcional

```python
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
import os

# 1. Modelo de datos
class AnalisisTexto(BaseModel):
    resumen: str = Field(description="Resumen breve del texto")
    sentimiento: str = Field(description="Sentimiento: Positivo, Neutro o Negativo")
    palabras_clave: list[str] = Field(description="3-5 palabras clave principales")

# 2. Parser
parser = PydanticOutputParser(pydantic_object=AnalisisTexto)

# 3. Prompt
prompt = PromptTemplate(
    template="""Analiza este texto cuidadosamente y proporciona un análisis estructurado:

{format_instructions}

TEXTO:
{texto}

ANÁLISIS:""",
    input_variables=["texto"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 4. LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# 5. Cadena
chain = prompt | llm | parser

# 6. Ejecución
if __name__ == "__main__":
    texto = "Me encantó la nueva película de acción, tiene efectos especiales increíbles."

    try:
        resultado = chain.invoke({"texto": texto})
        print("✅ Análisis exitoso:")
        print(resultado.model_dump_json(indent=2))
    except Exception as e:
        print(f"❌ Error: {e}")
```

---

## Conclusión

`PydanticOutputParser` sigue siendo una herramienta valiosa para obtener *structured output* en LangChain. Aunque requiere más código que los métodos modernos, te da control total sobre el proceso y funciona con cualquier modelo de lenguaje. Es especialmente útil cuando necesitas validaciones complejas o trabajas con modelos que no soportan *function calling* nativo.


In [35]:
from pydantic import BaseModel

class Usuario(BaseModel):
    id: int
    nombre: str
    activo: bool = True

data = {
    "id": 1,
    "nombre": "Juan Pérez"
}

usuario = Usuario(**data)

print(usuario)

id=1 nombre='Juan Pérez' activo=True


In [ ]:
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

In [37]:
load_dotenv(find_dotenv())
http_client = httpx.Client(verify=False)

In [38]:
class AnalisisTexto(BaseModel):
    resumen: str = Field(description="Resumen breve del texto.")
    sentimiento: str = Field(description="Sentimiento del texto (positivo, neutro o negativo)")

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0.7,
    http_client=http_client
)

In [40]:
structured_llm = llm.with_structured_output(AnalisisTexto)

In [41]:
texto_prueba = "Me encantó la nueva película de acción, tiene muchos efectos especiales y emoción."

resultado = structured_llm.invoke(f"Analiza el siguiente texto: {texto_prueba}")

d:\Repositorios\Github\curso_ia\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AnalisisTexto(resumen='El... sentimiento='positivo'), input_type=AnalisisTexto])
  return self.__pydantic_serializer__.to_python(


In [42]:
print(type(resultado))
print(dir(resultado))

<class '__main__.AnalisisTexto'>
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_extra_info__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic_root_model__', '__pydantic_

In [43]:
print(resultado.model_dump_json())

{"resumen":"El texto expresa una opinión positiva sobre una nueva película de acción, destacando sus efectos especiales y la emoción que genera.","sentimiento":"positivo"}
